# Phase 1.5: Exploratory Data Analysis (EDA) of Electricity Demand & Weather Covariates

**Project Title:** Electricity Demand Forecasting Using Deep Learning: A Comparative Study of 1D CNN, LSTM and CNN–LSTM Models  
**Forecast Horizon:** 15-Minute Ahead (Single-Step at 15-min cadence)  
**Granularity:** Individual Household / Meter Demand and Aggregate Cohort Demand  
**Experiments:** Consumption-Only vs. Consumption + Weather (Retrospective vs. Operational Availability)

---

## 1. Study Datasets Overview
1. **Dataset A (IIIT Delhi Faculty Housing):** 19 faculty apartments in Okhla, New Delhi (August 2013 – May 2014, 15-min resolution).
2. **Dataset B (CEEW Smart Meter Data):** Rural/semi-urban smart meters in Mathura (38 meters) and Bareilly (46 meters), UP (2019 – 2021, 3-min aggregated to 15-min).
3. **Weather Covariates:** ECMWF ERA5-Land Reanalysis at 0.1° resolution (Open-Meteo API) with retrospective as-of and 5-day operational availability lagging.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.size'] = 11
plt.rcParams['figure.titlesize'] = 13

DATA_DIR = Path('../data/processed')
METRICS_DIR = Path('../artifacts/metrics/eda')

## 2. Load Processed Feature Tables & Target Matrices

In [ ]:
delhi_feat = pd.read_parquet(DATA_DIR / 'delhi_features.parquet')
delhi_indiv = pd.read_parquet(DATA_DIR / 'delhi_individual_meters.parquet')

mathura_feat = pd.read_parquet(DATA_DIR / 'mathura_features.parquet')
mathura_indiv = pd.read_parquet(DATA_DIR / 'mathura_individual_meters.parquet')

bareilly_feat = pd.read_parquet(DATA_DIR / 'bareilly_features.parquet')
bareilly_indiv = pd.read_parquet(DATA_DIR / 'bareilly_individual_meters.parquet')

print(f"IIIT Delhi intervals:   {len(delhi_feat):,} rows, {len(delhi_feat.columns)} features")
print(f"CEEW Mathura intervals: {len(mathura_feat):,} rows, {len(mathura_feat.columns)} features")
print(f"CEEW Bareilly intervals:{len(bareilly_feat):,} rows, {len(bareilly_feat.columns)} features")

## 3. Cohort Availability & Target Validity Census

We inspect the distribution of active reporting meters and cohort validity across the continuous calendar grid.

In [ ]:
census_records = [
    {
        'Dataset': 'IIIT Delhi',
        'Total Intervals': len(delhi_feat),
        'Primary Valid (19/19)': delhi_feat['is_primary_cohort_valid'].sum(),
        'Sensitivity Valid (18/19)': delhi_feat['is_sensitivity_cohort_valid'].sum(),
        'Primary %': f"{delhi_feat['is_primary_cohort_valid'].mean()*100:.2f}%",
        'Sensitivity %': f"{delhi_feat['is_sensitivity_cohort_valid'].mean()*100:.2f}%",
    },
    {
        'Dataset': 'CEEW Mathura',
        'Total Intervals': len(mathura_feat),
        'Primary Valid (>=90%)': mathura_feat['is_primary_cohort_valid'].sum(),
        'Sensitivity Valid (85-89%)': mathura_feat['is_sensitivity_cohort_valid'].sum(),
        'Primary %': f"{mathura_feat['is_primary_cohort_valid'].mean()*100:.2f}%",
        'Sensitivity %': f"{mathura_feat['is_sensitivity_cohort_valid'].mean()*100:.2f}%",
    },
    {
        'Dataset': 'CEEW Bareilly',
        'Total Intervals': len(bareilly_feat),
        'Primary Valid (>=90%)': bareilly_feat['is_primary_cohort_valid'].sum(),
        'Sensitivity Valid (85-89%)': bareilly_feat['is_sensitivity_cohort_valid'].sum(),
        'Primary %': f"{bareilly_feat['is_primary_cohort_valid'].mean()*100:.2f}%",
        'Sensitivity %': f"{bareilly_feat['is_sensitivity_cohort_valid'].mean()*100:.2f}%",
    }
]
pd.DataFrame(census_records)

## 4. Aggregate Demand Summary Statistics (Average kW)

In [ ]:
stats_list = []
for name, df in [('IIIT Delhi (19 Apt)', delhi_feat), ('Mathura (>=90%)', mathura_feat), ('Bareilly (>=90%)', bareilly_feat)]:
    s = df['aggregate_demand_kw_primary'].dropna()
    stats_list.append({
        'Cohort': name,
        'Valid Count': len(s),
        'Mean (kW)': round(s.mean(), 2),
        'Std (kW)': round(s.std(), 2),
        'Min (kW)': round(s.min(), 2),
        '25% (kW)': round(s.quantile(0.25), 2),
        'Median (kW)': round(s.median(), 2),
        '75% (kW)': round(s.quantile(0.75), 2),
        'Max (kW)': round(s.max(), 2),
    })
pd.DataFrame(stats_list)

## 5. Diurnal and Weekly Load Profiles (Peak vs. Off-Peak)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for idx, (name, df) in enumerate([('IIIT Delhi', delhi_feat), ('CEEW Mathura', mathura_feat), ('CEEW Bareilly', bareilly_feat)]):
    ax = axes[idx]
    valid = df.dropna(subset=['aggregate_demand_kw_primary']).copy()
    if len(valid) > 0:
        valid['day_type'] = np.where(valid['is_weekend'] == 1, 'Weekend', 'Weekday')
        diurnal = valid.groupby(['hour_of_day', 'day_type'])['aggregate_demand_kw_primary'].mean().reset_index()
        sns.lineplot(data=diurnal, x='hour_of_day', y='aggregate_demand_kw_primary', hue='day_type', marker='o', ax=ax)
        ax.set_title(f"{name} Diurnal Curve")
        ax.set_xlabel('Hour of Day (IST)')
        ax.set_ylabel('Average Demand (kW)')
        ax.set_xticks(range(0, 24, 2))
plt.tight_layout()
plt.show()

## 6. Weather-Demand Non-Linearity (Temperature & Solar Radiation)

In [ ]:
from scipy.stats import pearsonr, spearmanr

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for idx, (name, df) in enumerate([('IIIT Delhi', delhi_feat), ('CEEW Mathura', mathura_feat), ('CEEW Bareilly', bareilly_feat)]):
    ax = axes[idx]
    col = 'aggregate_demand_kw_primary'
    valid_full = df.dropna(subset=[col, 'retro_temperature_2m'])
    total_n = len(valid_full)
    r_t, _ = pearsonr(valid_full['retro_temperature_2m'], valid_full[col])
    rho_t, _ = spearmanr(valid_full['retro_temperature_2m'], valid_full[col])

    valid = valid_full.sample(min(2500, total_n), random_state=42)
    sns.regplot(data=valid, x='retro_temperature_2m', y=col, ax=ax,
                scatter_kws={'alpha': 0.35, 's': 15, 'color': 'royalblue'},
                line_kws={'color': 'crimson', 'linewidth': 2, 'label': 'Exploratory Association (2nd-Order Fit)'},
                order=2)
    ax.set_title(f"{name}: Temp vs Demand\nPearson r = {r_t:+.3f} | Spearman ρ = {rho_t:+.3f} (N = {total_n:,})", fontsize=11, fontweight='bold')
    ax.set_xlabel('2m Temperature (°C)')
    ax.set_ylabel('Cohort Demand (kW)')
    ax.legend(loc='upper left', framealpha=0.9, fontsize=9)
    ax.grid(True, linestyle=':', alpha=0.5)

plt.tight_layout()
plt.show()


## 7. Contiguous Sequence Identification for Deep Learning (1D CNN, LSTM, CNN-LSTM)

Deep learning models require a continuous sliding lookback window $L$ (e.g., $L=16$ steps = 4 hours) predicting the next 15-minute horizon step ($H=1$). Lookback sequences MUST NOT cross missing intervals or blackout gaps.

In [ ]:
def count_contiguous_sequences(series, L=16, H=1):
    mask = series.notna().values
    W = L + H
    count = 0
    for i in range(len(mask) - W + 1):
        if mask[i : i + W].all():
            count += 1
    return count

seq_summary = []
for name, df in [('IIIT Delhi Primary (19 Apt)', delhi_feat['aggregate_demand_kw_primary']),
                 ('IIIT Delhi Sensitivity (18 Apt)', delhi_feat['aggregate_demand_kw_sensitivity']),
                 ('Mathura Primary (>=90%)', mathura_feat['aggregate_demand_kw_primary']),
                 ('Mathura Sensitivity (85-89%)', mathura_feat['aggregate_demand_kw_sensitivity']),
                 ('Bareilly Primary (>=90%)', bareilly_feat['aggregate_demand_kw_primary']),
                 ('Bareilly Sensitivity (85-89%)', bareilly_feat['aggregate_demand_kw_sensitivity'])]:
    n_seq = count_contiguous_sequences(df, L=16, H=1)
    seq_summary.append({'Series': name, 'Lookback L': 16, 'Horizon H': 1, 'Valid Sequences': n_seq})
pd.DataFrame(seq_summary)